In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


d:\NazamKalsi\damm\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def processPdf(path):
    documents = [];
    directory = Path(path)

    pdfFiles = list(directory.glob("**/*.pdf"))
    print(len(pdfFiles))

    # print("length: ",len(pdfFiles))

    for pdf in pdfFiles: 
        # print(pdf.name)

        try:
            loader = PyMuPDFLoader(str(pdf));
            doc = loader.load()          
            documents.extend(doc)
        except:
            pass
    return documents

docs = processPdf("../data");
# print(docs)

1


In [15]:
## test splitting
def createChunks(docs, chunkSize=1000,chunkOverlap=200):
    textSplitter = RecursiveCharacterTextSplitter(
        chunk_size=chunkSize,
        chunk_overlap= chunkOverlap,
        separators=['\n\n', '\n', ' ', ''],
        length_function=len
    )

    splitDocs = textSplitter.split_documents(docs)
    return splitDocs

chunks = createChunks(docs)
chunks

[Document(metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-03-12T00:20:34+11:00', 'source': '..\\data\\pdf.pdf', 'file_path': '..\\data\\pdf.pdf', 'total_pages': 77, 'format': 'PDF 1.5', 'title': 'The Metamorphosis', 'author': 'Franz Kafka', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'keywords': '', 'moddate': '2008-07-06T18:56:40+10:00', 'trapped': '', 'modDate': "D:20080706185640+10'00'", 'creationDate': "D:20080312002034+11'00'", 'page': 0}, page_content='Download free eBooks of classic literature, books and \nnovels at Planet eBook. Subscribe to our free eBooks blog \nand email newsletter.\nThe Metamorphosis\nBy Franz Kafka (1915)'),
 Document(metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-03-12T00:20:34+11:00', 'source': '..\\data\\pdf.pdf', 'file_path': '..\\data\\pdf.pdf', 'total_pages': 77, 'format': 'PDF 1.5', 'titl

In [7]:
# embedding

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [ ]:
class EmbeddingManager:
    def __init__(self, modelName = "all-MiniLM-L6-v2"):
        self.modelName = modelName
        self.model = None
        self._loadModel()
    # _ in start: protected function in python classes, only accessable in this class
    def _loadModel(self):
        try:
            self.model = SentenceTransformer(self.modelName)
            # print("model dimensions",self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print("error: ",e);
            raise


    def generateEmbedding(self,text:List[str])->np.ndarray:
        if not self.model:
            print("model not found")
            return;

        print("text length", len(text))
        embedding = self.model.encode(text,show_progress_bar=True)
        print(f"embedding shape {embedding.shape}")
        return embedding;

    def getEmbeddingDimensions(self) -> int:
        if not self.model:
            print("model not loaded");
        return self.model.get_sentence_embedding_dimension()


In [18]:
embeddingInstant = EmbeddingManager()
embeddingInstant

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6329.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model dimensions 384


In [27]:
class VectorStore:
    def __init__(self, collectionName = "pdfDocs", persistDir = "../data/vectorStore"):
        self.collectionName = collectionName
        self.persistDir = persistDir
        self.client = None
        self.collection = None
        self._initializeStore_()

    def _initializeStore_(self):
        try:
            os.makedirs(self.persistDir, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.collectionName)        
            self.collection =self.client.get_or_create_collection(name=self.collectionName,metadata={"description": "PDF for RAG"})

            print("vector store initialize")
        except Exception as e:
            print(e)

    def addDocs(self, docs:List[Any], embedding:np.ndarray):
        ids = []
        metaDatas = []
        docText= []
        embeddingList = []

        for i, (doc, embedding) in enumerate(zip(docs, embedding)):
            docId = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(docId)

            metaData = dict(doc.metadata)
            metaData['doc_index']= i
            metaData["content_length"] = len(doc.page_content)

            docText.append(doc.page_content)
            embeddingList.append(embedding.tolist())


        try:
            self.collection.add(
                ids=ids,
                embedding = embeddingList,
                metaDatas=metaDatas,
                documents = docText
            )

            print("doc length", len(doc))

        except Exception as e:
             pass


vectorStoreInstance = VectorStore()



vector store initialize


In [28]:
texts = [doc.page_content for doc in chunks]
texts
embeddings = embeddingInstant.generateEmbedding(texts)
vectorS = vectorStoreInstance.addDocs(chunks, embeddings)
vectorS


text length 152


Batches: 100%|██████████| 5/5 [00:05<00:00,  1.17s/it]

embedding shape (152, 384)
